# CNN + GloVe (starter) — Jigsaw toxic comments

**Preprocessing:** `preprocess_for_cnn` (word indices, train-only vocab).

**Default in this notebook:** **full baseline** — entire train split (after stratified train/val), `MAX_LEN = 256`, multi-epoch training, no row caps. Other modes remain available: **quick** smoke (`QUICK_ITERATION = True`) or **progress report** (`PROGRESS_REPORT_MODE = True`).

**Next steps for your proposal:** Download GloVe (e.g. `glove.6B.100d.txt`), implement `build_glove_weight_matrix(vocab, path, dim)` aligned to `data.vocab`, pass `embedding.weight.data.copy_(...)`. Until then this notebook uses **trainable** random embeddings so the pipeline runs.

**Metrics (course proposal):** micro/macro F1, precision/recall/ROC-AUC per label, confusion matrices, training time, parameter count. Validation predictions and metrics are saved under `notebooks/cnn_baseline_outputs/`: `cnn_baseline_*` when `USE_POS_WEIGHT=False`, `cnn_posweight_*` when `USE_POS_WEIGHT=True` (train-only `pos_weight` in `BCEWithLogitsLoss`). With `USE_POS_WEIGHT=True`, the eval cell also runs **per-label threshold tuning** on the validation set (max F1 per label, not 0.5), prints a three-way comparison vs baseline `@0.5`, and stores `threshold_tuned_eval` plus `y_pred_tuned` / `best_threshold_per_label` in the artifacts.

In [1]:
from pathlib import Path
import sys

# Repo root (contains preprocessing/)
ROOT = Path.cwd().resolve()
for c in (ROOT, ROOT.parent):
    if (c / "preprocessing" / "text_preprocessing.py").exists():
        ROOT = c
        break
sys.path.insert(0, str(ROOT))
# notebooks/ for metrics_helpers
sys.path.insert(0, str(ROOT / "notebooks"))

In [ ]:
import inspect
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset

from preprocessing.text_preprocessing import LABEL_COLUMNS
from sklearn.metrics import f1_score as sk_f1

from metrics_helpers import multilabel_evaluation_report, per_label_confusion_matrices, torch_parameter_count

def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = pick_device()
RNG = np.random.default_rng(42)
torch.manual_seed(42)

from preprocessing.text_preprocessing import preprocess_for_cnn

# ========== Run configuration (edit here) ==========
# Three modes (quick wins if True):
#   1) quick smoke     — QUICK_ITERATION = True
#   2) progress report — QUICK_ITERATION = False, PROGRESS_REPORT_MODE = True
#   3) full              — QUICK_ITERATION = False, PROGRESS_REPORT_MODE = False
QUICK_ITERATION = False
PROGRESS_REPORT_MODE = False

# Class imbalance: BCEWithLogitsLoss(pos_weight) from train-only n_neg/n_pos per label. False = baseline loss.
USE_POS_WEIGHT = True

# Optional pretrained embedding init (controlled comparison vs random embedding baseline).
USE_GLOVE = True
GLOVE_DIM = 100
GLOVE_PATH = ROOT / "data" / "glove.6B.100d.txt"
GLOVE_TRAINABLE = True

# Saved basename tag (distinct from Macro-F1–selected earlystop artifacts). Prefer "lossselect" for min-val-loss selection.
OUTPUT_FILE_TAG = "lossselect"

LR = 1e-3
MIN_DELTA = 0.005
PATIENCE = 2

# New preprocessing API: ALL ON mode for the current full run.
USE_ITERATIVE_STRATIFY = True
REBALANCE_TRAIN = True
CLEAN_TO_TOXIC_RATIO = 3.0
RARE_LABELS = ("severe_toxic", "threat", "identity_hate")
RARE_OVERSAMPLE_FACTOR = 2.0
MAX_COPIES_PER_ROW = 3
REBALANCE_RANDOM_STATE = 42
PRINT_PREPROCESSING_DIAGNOSTICS = True

# --- Quick smoke-test (used only if QUICK_ITERATION) ---
QUICK_MAX_TRAIN_SAMPLES = 2048
QUICK_MAX_VAL_SAMPLES = 512
QUICK_MAX_LEN = 64
QUICK_BATCH_SIZE = 128
QUICK_EPOCHS = 1
QUICK_MAX_VOCAB = 3000
QUICK_MIN_FREQ = 1

# --- Progress-report run (used if not quick and PROGRESS_REPORT_MODE) ---
PROGRESS_MAX_TRAIN_SAMPLES = 10_000
PROGRESS_MAX_VAL_SAMPLES = 2_000
PROGRESS_MAX_LEN = 128
PROGRESS_BATCH_SIZE = 64
PROGRESS_EPOCHS = 2
PROGRESS_MAX_VOCAB = 10_000
PROGRESS_MIN_FREQ = 2

# --- Full run (both flags False) ---
FULL_MAX_TRAIN_SAMPLES = None
FULL_MAX_VAL_SAMPLES = None
FULL_MAX_LEN = 256
FULL_BATCH_SIZE = 64
FULL_EPOCHS = 20
FULL_MAX_VOCAB = 50_000
FULL_MIN_FREQ = 2

if QUICK_ITERATION:
    run_mode = "quick (smoke test)"
    _train_n = QUICK_MAX_TRAIN_SAMPLES
    _val_n = QUICK_MAX_VAL_SAMPLES
    MAX_LEN = QUICK_MAX_LEN
    BATCH_SIZE = QUICK_BATCH_SIZE
    EPOCHS = QUICK_EPOCHS
    MAX_VOCAB = QUICK_MAX_VOCAB
    MIN_FREQ = QUICK_MIN_FREQ
elif PROGRESS_REPORT_MODE:
    run_mode = "progress report"
    _train_n = PROGRESS_MAX_TRAIN_SAMPLES
    _val_n = PROGRESS_MAX_VAL_SAMPLES
    MAX_LEN = PROGRESS_MAX_LEN
    BATCH_SIZE = PROGRESS_BATCH_SIZE
    EPOCHS = PROGRESS_EPOCHS
    MAX_VOCAB = PROGRESS_MAX_VOCAB
    MIN_FREQ = PROGRESS_MIN_FREQ
else:
    run_mode = "full"
    _train_n = FULL_MAX_TRAIN_SAMPLES
    _val_n = FULL_MAX_VAL_SAMPLES
    MAX_LEN = FULL_MAX_LEN
    BATCH_SIZE = FULL_BATCH_SIZE
    EPOCHS = FULL_EPOCHS
    MAX_VOCAB = FULL_MAX_VOCAB
    MIN_FREQ = FULL_MIN_FREQ

data = preprocess_for_cnn(
    max_len=MAX_LEN,
    validation_fraction=0.1,
    random_state=42,
    min_freq=MIN_FREQ,
    max_vocab=MAX_VOCAB,
    max_train_samples=_train_n,
    max_val_samples=_val_n,
    use_iterative_stratify=USE_ITERATIVE_STRATIFY,
    rebalance_train=REBALANCE_TRAIN,
    clean_to_toxic_ratio=CLEAN_TO_TOXIC_RATIO,
    rare_labels=RARE_LABELS,
    rare_oversample_factor=RARE_OVERSAMPLE_FACTOR,
    max_copies_per_row=MAX_COPIES_PER_ROW,
    rebalance_random_state=REBALANCE_RANDOM_STATE,
    print_diagnostics=PRINT_PREPROCESSING_DIAGNOSTICS,
)
vocab_size = len(data.vocab)
embed_dim = GLOVE_DIM if USE_GLOVE else 100
num_labels = len(LABEL_COLUMNS)

# Train-only pos_weight = n_negative / n_positive per label (for optional BCEWithLogitsLoss).
_y_train = np.asarray(data.y_train, dtype=np.float64)
pos_weight_train = np.zeros(num_labels, dtype=np.float64)
for _i in range(num_labels):
    _pos = float(_y_train[:, _i].sum())
    _neg = float(_y_train.shape[0] - _pos)
    pos_weight_train[_i] = (_neg / _pos) if _pos > 0 else 1.0
POS_WEIGHT = (
    torch.tensor(pos_weight_train, dtype=torch.float32, device=DEVICE)
    if USE_POS_WEIGHT
    else None
)

def build_glove_weight_matrix(
    vocab: dict[str, int],
    glove_path: Path,
    dim: int,
    *,
    rng_seed: int = 42,
) -> tuple[np.ndarray, int, int]:
    if not glove_path.is_file():
        raise FileNotFoundError(
            f"GloVe file not found: {glove_path}. Set GLOVE_PATH or place glove.6B.100d.txt under data/."
        )

    rng = np.random.default_rng(rng_seed)
    matrix = rng.normal(loc=0.0, scale=0.05, size=(len(vocab), dim)).astype(np.float32)

    pad_idx = vocab.get("<pad>", 0)
    matrix[pad_idx] = 0.0

    matched = 0
    with glove_path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != dim + 1:
                continue
            word = parts[0]
            idx = vocab.get(word)
            if idx is None:
                continue
            matrix[idx] = np.asarray(parts[1:], dtype=np.float32)
            matched += 1

    total = len(vocab)
    return matrix, matched, total


glove_stats = {
    "enabled": bool(USE_GLOVE),
    "path": str(GLOVE_PATH),
    "dim": int(embed_dim),
    "trainable": bool(GLOVE_TRAINABLE),
    "matched_words": None,
    "vocab_size": int(vocab_size),
    "coverage_pct": None,
}
glove_weight_matrix = None
if USE_GLOVE:
    glove_weight_matrix, _matched, _total = build_glove_weight_matrix(
        data.vocab, Path(GLOVE_PATH), embed_dim
    )
    glove_stats["matched_words"] = int(_matched)
    glove_stats["vocab_size"] = int(_total)
    glove_stats["coverage_pct"] = float((100.0 * _matched / _total) if _total > 0 else 0.0)

n_train, n_val = data.X_train.shape[0], data.X_val.shape[0]
print("=" * 60)
print("CONFIG")
print("  mode:", run_mode)
print("  device:", DEVICE)
print("  USE_POS_WEIGHT:", USE_POS_WEIGHT)
print("  USE_GLOVE:", USE_GLOVE)
print(
    "  PREPROCESSING:",
    f"use_iterative_stratify={USE_ITERATIVE_STRATIFY}",
    f"rebalance_train={REBALANCE_TRAIN}",
    f"clean_to_toxic_ratio={CLEAN_TO_TOXIC_RATIO}",
    f"rare_oversample_factor={RARE_OVERSAMPLE_FACTOR}",
)
if USE_GLOVE:
    print("  GLOVE_PATH:", GLOVE_PATH)
    print(
        "  GLOVE_COVERAGE:",
        f"{glove_stats['matched_words']}/{glove_stats['vocab_size']} ({glove_stats['coverage_pct']:.2f}%)",
    )
print("  train_samples:", n_train, "| val_samples:", n_val)
print("  MAX_LEN:", MAX_LEN, "| vocab_size:", vocab_size, "| embed_dim:", embed_dim)
print("  BATCH_SIZE:", BATCH_SIZE, "| EPOCHS:", EPOCHS)
print(
    "  EARLY_STOP (val-loss): checkpoint if val_loss decreases by >= ",
    MIN_DELTA,
    "; patience:",
    PATIENCE,
)
print("=" * 60)

CONFIG
  mode: full
  device: cpu
  USE_POS_WEIGHT: True
  USE_GLOVE: True
  PREPROCESSING: use_iterative_stratify=False rebalance_train=False
  GLOVE_PATH: C:\Users\alici\Downloads\cmpe258-2026Fall-ToxicCommentDetection\data\glove.6B.100d.txt
  GLOVE_COVERAGE: 17881/50000 (35.76%)
  train_samples: 143613 | val_samples: 15958
  MAX_LEN: 256 | vocab_size: 50000 | embed_dim: 100
  BATCH_SIZE: 64 | EPOCHS: 20
  EARLY_STOP (val-loss): checkpoint if val_loss decreases by >=  0.005 ; patience: 2


In [3]:
class TextCNN(nn.Module):
    # Kim-style multi-channel CNN over embeddings

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        num_labels: int,
        padding_idx: int = 0,
        kernel_sizes: tuple[int, ...] = (3, 4, 5),
        num_filters: int = 100,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes])
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_labels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e = self.embedding(x).transpose(1, 2)
        pools = []
        for conv in self.convs:
            h = torch.relu(conv(e))
            pools.append(h.max(dim=2).values)
        h = torch.cat(pools, dim=1)
        return self.fc(self.dropout(h))


model = TextCNN(vocab_size, embed_dim, num_labels, padding_idx=0).to(DEVICE)
if USE_GLOVE:
    if glove_weight_matrix is None:
        raise RuntimeError("USE_GLOVE=True but glove_weight_matrix is not initialized.")
    with torch.no_grad():
        model.embedding.weight.copy_(torch.tensor(glove_weight_matrix, dtype=torch.float32, device=DEVICE))
    model.embedding.weight.requires_grad = bool(GLOVE_TRAINABLE)

if USE_POS_WEIGHT:
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
else:
    loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(
    "Embedding init:",
    "GloVe" if USE_GLOVE else "random trainable",
    flush=True,
)
if USE_GLOVE:
    print(
        "  GloVe coverage:",
        f"{glove_stats['matched_words']}/{glove_stats['vocab_size']} ({glove_stats['coverage_pct']:.2f}%)",
        flush=True,
    )
    print("  Embedding trainable:", bool(GLOVE_TRAINABLE), flush=True)

print(
    "Training loss: BCEWithLogitsLoss —",
    "pos_weight ON (train-only n_neg/n_pos per label)"
    if USE_POS_WEIGHT
    else "pos_weight OFF (baseline)",
    flush=True,
)
if USE_POS_WEIGHT:
    for _name, _w in zip(LABEL_COLUMNS, pos_weight_train):
        print(f"  {_name}: {_w:.4f}", flush=True)

train_ds = TensorDataset(
    torch.tensor(data.X_train, dtype=torch.long),
    torch.tensor(data.y_train, dtype=torch.float32),
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

val_ds = TensorDataset(
    torch.tensor(data.X_val, dtype=torch.long),
    torch.tensor(data.y_val, dtype=torch.float32),
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

def evaluate_val_epoch(curr_model: nn.Module) -> tuple[float, float, float, float, dict[str, float]]:
    curr_model.eval()
    val_loss_sum = 0.0
    val_batches = 0
    logits_out = []

    infer_t0 = time.perf_counter()
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = curr_model(xb)
            val_loss_sum += loss_fn(logits, yb).item()
            val_batches += 1
            logits_out.append(logits.cpu().numpy())
    prob_val_epoch = torch.sigmoid(torch.from_numpy(np.concatenate(logits_out, axis=0))).numpy()
    infer_s = time.perf_counter() - infer_t0

    pred_val_epoch = (prob_val_epoch >= 0.5).astype(np.int64)
    _, summary_epoch = multilabel_evaluation_report(
        data.y_val, pred_val_epoch, prob_val_epoch, LABEL_COLUMNS
    )
    per_label_f1_epoch = {
        str(lbl): float(sk_f1(data.y_val[:, i], pred_val_epoch[:, i], zero_division=0))
        for i, lbl in enumerate(LABEL_COLUMNS)
    }
    val_loss_epoch = val_loss_sum / max(val_batches, 1)
    return val_loss_epoch, float(summary_epoch["f1_micro"]), float(summary_epoch["f1_macro"]), infer_s, per_label_f1_epoch


t0 = time.perf_counter()
epoch_history = []
best_epoch = 0
best_checkpoint_val_loss = float("inf")  # lowest val_loss seen among saved checkpoints
best_state_dict = None
best_train_loss = None
best_micro_f1 = None
best_macro_f1_epoch = None
best_val_inference_time_s = None
patience_bad_epochs = 0

for epoch in range(EPOCHS):
    model.train()
    epoch_t0 = time.perf_counter()
    epoch_loss = 0.0
    n_batches = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1

    train_loss_epoch = epoch_loss / max(n_batches, 1)
    val_loss_epoch, micro_f1_epoch, macro_f1_epoch, infer_s_epoch, per_label_f1_epoch = evaluate_val_epoch(model)
    train_s_epoch = time.perf_counter() - epoch_t0

    epoch_history.append(
        {
            "epoch": int(epoch + 1),
            "train_loss": float(train_loss_epoch),
            "val_loss": float(val_loss_epoch),
            "micro_f1": float(micro_f1_epoch),
            "macro_f1": float(macro_f1_epoch),
            "inference_time_s": float(infer_s_epoch),
            "per_label_f1": {k: float(v) for k, v in per_label_f1_epoch.items()},
        }
    )

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | train_loss={train_loss_epoch:.4f} | val_loss={val_loss_epoch:.4f} | "
        f"micro_f1={micro_f1_epoch:.4f} | macro_f1={macro_f1_epoch:.4f} | train_s={train_s_epoch:.1f} | infer_s={infer_s_epoch:.1f}",
        flush=True,
    )

    improved = (epoch == 0) or (val_loss_epoch < best_checkpoint_val_loss - MIN_DELTA)
    if improved:
        best_epoch = epoch + 1
        best_checkpoint_val_loss = float(val_loss_epoch)
        best_micro_f1 = float(micro_f1_epoch)
        best_macro_f1_epoch = float(macro_f1_epoch)
        best_train_loss = float(train_loss_epoch)
        best_val_inference_time_s = float(infer_s_epoch)
        best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_bad_epochs = 0
        print(
            f"  -> New best epoch (best val_loss so far={best_checkpoint_val_loss:.4f}, min_delta={MIN_DELTA:.4f})",
            flush=True,
        )
    else:
        patience_bad_epochs += 1
        print(
            f"  -> Val loss did not improve by >= {MIN_DELTA:.4f} (best_val_loss={best_checkpoint_val_loss:.4f}; bad_epochs={patience_bad_epochs}/{PATIENCE})",
            flush=True,
        )
        if patience_bad_epochs >= PATIENCE:
            print("Early stopping triggered.", flush=True)
            break

for _eh in epoch_history:
    _eh["is_best_epoch"] = bool(int(_eh["epoch"]) == int(best_epoch))

train_seconds = time.perf_counter() - t0
if best_state_dict is None:
    raise RuntimeError("Best model checkpoint not found; training did not produce any epoch.")
model.load_state_dict(best_state_dict)

final_train_loss = float(best_train_loss)
val_loss = float(best_checkpoint_val_loss)
best_metrics_05 = {
    "micro_f1": float(best_micro_f1),
    "macro_f1": float(best_macro_f1_epoch),
}

print(f"Training wall time: {train_seconds:.1f} s")
print(
    f"Checkpoint selection: lowest validation loss (early stop requires >= {MIN_DELTA:.4f} improvement vs prior best)."
)
print(
    f"Best epoch (by val_loss): {best_epoch} | train_loss: {final_train_loss:.4f} | best val_loss: {val_loss:.4f} | "
    f"@0.5 micro_f1={best_metrics_05['micro_f1']:.4f} | macro_f1={best_metrics_05['macro_f1']:.4f} | val infer_s={best_val_inference_time_s:.2f}"
)

# Per-epoch history table for notebook inspection
history_df = pd.DataFrame(epoch_history)
if "per_label_f1" in history_df.columns:
    per_label_df = history_df.pop("per_label_f1").apply(pd.Series)
    per_label_df = per_label_df.add_prefix("f1_")
    history_df = pd.concat([history_df, per_label_df], axis=1)

print()
print("Epoch history (is_best_epoch=True on the checkpoint / lowest-val-loss epoch):")
display(history_df)

Embedding init: GloVe
  GloVe coverage: 17881/50000 (35.76%)
  Embedding trainable: True
Training loss: BCEWithLogitsLoss — pos_weight ON (train-only n_neg/n_pos per label)
  toxic: 9.4423
  severe_toxic: 99.2884
  obscene: 17.8493
  threat: 336.9129
  insult: 19.2443
  identity_hate: 112.7078
Epoch 1/20 | train_loss=0.5579 | val_loss=0.4135 | micro_f1=0.4450 | macro_f1=0.3697 | train_s=462.4 | infer_s=16.4
  -> New best epoch (best val_loss so far=0.4135, min_delta=0.0050)
Epoch 2/20 | train_loss=0.3466 | val_loss=0.4035 | micro_f1=0.4967 | macro_f1=0.4043 | train_s=403.0 | infer_s=14.9
  -> New best epoch (best val_loss so far=0.4035, min_delta=0.0050)
Epoch 3/20 | train_loss=0.2764 | val_loss=0.4598 | micro_f1=0.5626 | macro_f1=0.4490 | train_s=442.8 | infer_s=12.5
  -> Val loss did not improve by >= 0.0050 (best_val_loss=0.4035; bad_epochs=1/2)
Epoch 4/20 | train_loss=0.2329 | val_loss=0.5357 | micro_f1=0.5662 | macro_f1=0.4494 | train_s=402.4 | infer_s=12.2
  -> Val loss did not i

,epoch,train_loss,val_loss,micro_f1,macro_f1,inference_time_s,is_best_epoch,f1_toxic,f1_severe_toxic,f1_obscene,f1_threat,f1_insult,f1_identity_hate
0,1,0.557909,0.413494,0.445006,0.369691,16.422895,False,0.648520,0.229345,0.583714,0.057627,0.548852,0.150089
1,2,0.346561,0.403532,0.496679,0.404303,14.933857,True,0.653111,0.274194,0.642225,0.075277,0.590756,0.190255
2,3,0.276444,0.459763,0.562588,0.449045,12.479962,False,0.700081,0.336770,0.696694,0.145800,0.619963,0.194963
3,4,0.232861,0.535722,0.566213,0.449448,12.203450,False,0.676931,0.339265,0.647414,0.191489,0.631928,0.209663


In [4]:
import json

# Uses sk_f1 imported in the config/training cell


def predict_logits(model, X_np: np.ndarray, batch_size: int = 512) -> np.ndarray:
    model.eval()
    out = []
    x = torch.tensor(X_np, dtype=torch.long)
    with torch.no_grad():
        for i in range(0, len(x), batch_size):
            logits = model(x[i : i + batch_size].to(DEVICE))
            out.append(logits.cpu().numpy())
    return np.concatenate(out, axis=0)


# Raw validation inference time (model forward + sigmoid only; excludes threshold search)
infer_t0 = time.perf_counter()
logits_val = predict_logits(model, data.X_val)
prob_val = torch.sigmoid(torch.from_numpy(logits_val)).numpy()
inference_time_s = time.perf_counter() - infer_t0
pred_val_05 = (prob_val >= 0.5).astype(np.int64)
pred_val = pred_val_05

y_val_arr = np.asarray(data.y_val, dtype=np.float32)

per_label_df_05, summary_05 = multilabel_evaluation_report(
    y_val_arr, pred_val_05, prob_val, LABEL_COLUMNS
)
per_label_df, summary = per_label_df_05, summary_05
display(per_label_df_05)
confs = per_label_confusion_matrices(y_val_arr, pred_val_05, LABEL_COLUMNS)
for name, m in confs.items():
    print(name, "\n", m)

print()
if USE_POS_WEIGHT and USE_GLOVE:
    _val_title = "CNN + GloVe + pos_weight"
elif USE_POS_WEIGHT:
    _val_title = "CNN + pos_weight"
elif USE_GLOVE:
    _val_title = "CNN + GloVe"
else:
    _val_title = "CNN baseline"
print(f"=== Validation @ 0.5 threshold — {_val_title} === (best checkpoint chosen by MIN val_loss) ===")
print(f"  Total training time: {train_seconds:.2f} s")
print(f"  Best epoch: {best_epoch} / {len(epoch_history)}")
print(f"  Best-epoch val inference_s: {best_val_inference_time_s:.2f}")
print(f"  Micro F1:  {summary_05['f1_micro']:.4f}")
print(f"  Macro F1: {summary_05['f1_macro']:.4f}")
print("  Per-label F1:")
for _, row in per_label_df_05.iterrows():
    print(f"    {row['label']}: {row['f1']:.4f}")

print()
print("--- Proposal summary (record for report / comparison) ---")
print(f"  device: {DEVICE}")
print(f"  training_time_s: {train_seconds:.2f}")
print(f"  inference_time_s: {inference_time_s:.2f}")
print(f"  num_parameters: {torch_parameter_count(model)}")
if summary_05["f1_macro"] == 0.0 and summary_05["f1_micro"] == 0.0:
    print(
        "  Note: F1 can be 0 if every predicted label is 0 at threshold 0.5 (common with very few epochs)."
        " ROC-AUC may still be > 0.5. Use more EPOCHS or check data / logits scale."
    )

print()
print("--- Run summary ---")
print(f"  mode: {run_mode}")
print(f"  best_epoch: {best_epoch}")
print(f"  final_train_loss: {final_train_loss:.4f}")
print(f"  val_loss: {val_loss:.4f}")
print(f"  inference_time_s: {inference_time_s:.2f}")
print(f"  best_epoch_val_inference_time_s: {best_val_inference_time_s:.2f}")
if USE_GLOVE:
    print(
        "  glove_coverage:",
        f"{glove_stats['matched_words']}/{glove_stats['vocab_size']} ({glove_stats['coverage_pct']:.2f}%)",
    )

tuned_threshold_eval = None
pred_val_tuned = None
best_thresholds_list = None
threshold_tuning_time_s = None

if USE_POS_WEIGHT:
    tune_t0 = time.perf_counter()
    y_int = y_val_arr.astype(np.int64)
    ths = np.linspace(0.01, 0.99, 99)
    best_meta = []
    pred_tuned = np.zeros_like(y_int, dtype=np.int64)
    for i, lbl in enumerate(LABEL_COLUMNS):
        yt = y_int[:, i]
        yp_col = prob_val[:, i]
        best_f, best_th = -1.0, 0.5
        for t in ths:
            pr = (yp_col >= t).astype(np.int64)
            f = sk_f1(yt, pr, zero_division=0)
            if f > best_f + 1e-12 or (
                abs(f - best_f) <= 1e-12 and abs(float(t) - 0.5) < abs(best_th - 0.5)
            ):
                best_f, best_th = float(f), float(t)
        best_meta.append({"label": lbl, "threshold": best_th})
        pred_tuned[:, i] = (yp_col >= best_th).astype(np.int64)

    per_label_df_tuned, summary_tuned = multilabel_evaluation_report(
        y_val_arr, pred_tuned, prob_val, LABEL_COLUMNS
    )
    pred_val_tuned = pred_tuned

    print()
    print("=== Per-label threshold tuning (val, maximize F1 per label; not 0.5) ===")
    for bm, (_, row) in zip(best_meta, per_label_df_tuned.iterrows()):
        print(
            f"  {bm['label']}: threshold={bm['threshold']:.4f} | "
            f"F1={float(row['f1']):.4f} P={float(row['precision']):.4f} R={float(row['recall']):.4f}"
        )
    print(f"  Micro F1 (tuned):  {summary_tuned['f1_micro']:.4f}")
    print(f"  Macro F1 (tuned): {summary_tuned['f1_macro']:.4f}")

    _baseline_json = ROOT / "notebooks" / "cnn_baseline_outputs" / "cnn_baseline_metrics.json"
    if _baseline_json.is_file():
        _bm = json.loads(_baseline_json.read_text(encoding="utf-8"))
        print()
        print("--- Comparison (validation) ---")
        print(
            f"  Baseline CNN @0.5:        micro {_bm['f1_micro']:.4f} | macro {_bm['f1_macro']:.4f}"
        )
        print(
            f"  pos_weight CNN @0.5:      micro {summary_05['f1_micro']:.4f} | macro {summary_05['f1_macro']:.4f}"
        )
        print(
            f"  pos_weight + tuned thr:   micro {summary_tuned['f1_micro']:.4f} | macro {summary_tuned['f1_macro']:.4f}"
        )

    threshold_tuning_time_s = time.perf_counter() - tune_t0
    print(f"  Threshold tuning time: {threshold_tuning_time_s:.2f} s")

    tuned_threshold_eval = {
        "best_threshold_per_label": {b["label"]: float(b["threshold"]) for b in best_meta},
        "f1_micro": float(summary_tuned["f1_micro"]),
        "f1_macro": float(summary_tuned["f1_macro"]),
        "threshold_tuning_time_s": float(threshold_tuning_time_s),
        "per_label": [
            {
                "label": str(row["label"]),
                "precision": float(row["precision"]),
                "recall": float(row["recall"]),
                "f1": float(row["f1"]),
                "roc_auc": float(row["roc_auc"]),
            }
            for _, row in per_label_df_tuned.iterrows()
        ],
    }
    best_thresholds_list = [float(b["threshold"]) for b in best_meta]

# Save validation predictions + metrics for later comparison
out_dir = ROOT / "notebooks" / "cnn_baseline_outputs"
out_dir.mkdir(parents=True, exist_ok=True)
if USE_POS_WEIGHT and USE_GLOVE:
    _artifact_stem = f"cnn_glove_posweight_{OUTPUT_FILE_TAG}"
elif USE_POS_WEIGHT:
    _artifact_stem = f"cnn_posweight_{OUTPUT_FILE_TAG}"
elif USE_GLOVE:
    _artifact_stem = "cnn_glove_baseline"
else:
    _artifact_stem = "cnn_baseline"
val_npz = out_dir / f"{_artifact_stem}_val.npz"
_save_npz = dict(
    y_true=y_val_arr,
    y_pred=pred_val_05.astype(np.int64),
    y_prob=prob_val.astype(np.float32),
    label_names=np.array(LABEL_COLUMNS, dtype=object),
)
if USE_POS_WEIGHT and pred_val_tuned is not None:
    _save_npz["y_pred_tuned"] = pred_val_tuned.astype(np.int64)
    _save_npz["best_threshold_per_label"] = np.array(
        best_thresholds_list, dtype=np.float32
    )
np.savez_compressed(val_npz, **_save_npz)
metrics_export = {
    "checkpoint_selection": {
        "criterion": "min_validation_loss",
        "min_absolute_improvement": float(MIN_DELTA),
        "patience_bad_epochs_max": int(PATIENCE),
    },
    "model": "TextCNN",
    "loss": "BCEWithLogitsLoss",
    "use_pos_weight": bool(USE_POS_WEIGHT),
    "use_glove": bool(USE_GLOVE),
    "glove": {
        "path": str(GLOVE_PATH),
        "dim": int(embed_dim),
        "trainable": bool(GLOVE_TRAINABLE),
        "matched_words": glove_stats["matched_words"],
        "vocab_size": glove_stats["vocab_size"],
        "coverage_pct": glove_stats["coverage_pct"],
    },
    "pos_weight_per_label": {
        str(LABEL_COLUMNS[i]): float(pos_weight_train[i]) for i in range(len(LABEL_COLUMNS))
    },
    "run_mode": run_mode,
    "device": str(DEVICE),
    "training_time_s": float(train_seconds),
    "inference_time_s": float(inference_time_s),
    "best_epoch": int(best_epoch),
    "epochs": int(EPOCHS),
    "epochs_ran": int(len(epoch_history)),
    "best_epoch_val_inference_time_s": float(best_val_inference_time_s),
    "config": {
        "max_len": int(MAX_LEN),
        "batch_size": int(BATCH_SIZE),
        "lr": float(LR),
        "min_freq": int(MIN_FREQ),
        "max_vocab": int(MAX_VOCAB),
        "validation_fraction": 0.1,
        "random_state": 42,
    },
    "train_samples": int(n_train),
    "val_samples": int(n_val),
    "final_train_loss": float(final_train_loss),
    "val_loss": float(val_loss),
    "f1_micro": float(summary_05["f1_micro"]),
    "f1_macro": float(summary_05["f1_macro"]),
    "per_label_f1": {
        str(row["label"]): float(row["f1"]) for _, row in per_label_df_05.iterrows()
    },
    "per_label": [
        {
            "label": str(row["label"]),
            "precision": float(row["precision"]),
            "recall": float(row["recall"]),
            "f1": float(row["f1"]),
            "roc_auc": float(row["roc_auc"]),
        }
        for _, row in per_label_df_05.iterrows()
    ],
    "num_parameters": int(torch_parameter_count(model)),
    "epoch_history": epoch_history,
}
if USE_POS_WEIGHT and tuned_threshold_eval is not None:
    metrics_export["threshold_tuning_time_s"] = float(threshold_tuning_time_s)
    metrics_export["threshold_tuned_eval"] = tuned_threshold_eval

if USE_GLOVE:
    _random_metrics = ROOT / "notebooks" / "cnn_baseline_outputs" / "cnn_posweight_earlystop_metrics.json"
    if _random_metrics.is_file():
        _rm = json.loads(_random_metrics.read_text(encoding="utf-8"))
        print()
        print("--- Comparison vs random embedding (same pos_weight + earlystop setup) ---")
        print(f"  random @0.5:  micro {_rm['f1_micro']:.4f} | macro {_rm['f1_macro']:.4f}")
        print(f"  glove  @0.5:  micro {summary_05['f1_micro']:.4f} | macro {summary_05['f1_macro']:.4f}")
        if "threshold_tuned_eval" in _rm and tuned_threshold_eval is not None:
            print(
                f"  random tuned: micro {_rm['threshold_tuned_eval']['f1_micro']:.4f} | "
                f"macro {_rm['threshold_tuned_eval']['f1_macro']:.4f}"
            )
            print(
                f"  glove tuned:  micro {tuned_threshold_eval['f1_micro']:.4f} | "
                f"macro {tuned_threshold_eval['f1_macro']:.4f}"
            )

metrics_json = out_dir / f"{_artifact_stem}_metrics.json"
with open(metrics_json, "w", encoding="utf-8") as f:
    json.dump(metrics_export, f, indent=2)
print()
print("Saved:", val_npz)
print("Saved:", metrics_json)

,label,precision,recall,f1,roc_auc
0,toxic,0.520849,0.875406,0.653111,0.962537
1,severe_toxic,0.160546,0.938650,0.274194,0.981456
2,obscene,0.493843,0.918072,0.642225,0.979952
3,threat,0.039171,0.962264,0.075277,0.985645
4,insult,0.440200,0.897829,0.590756,0.972064
5,identity_hate,0.106864,0.866197,0.190255,0.962584


toxic 
 [[13176  1241]
 [  192  1349]]
severe_toxic 
 [[14995   800]
 [   10   153]]
obscene 
 [[14347   781]
 [   68   762]]
threat 
 [[14654  1251]
 [    2    51]]
insult 
 [[14281   894]
 [   80   703]]
identity_hate 
 [[14788  1028]
 [   19   123]]

=== Validation @ 0.5 threshold — CNN + GloVe + pos_weight === (best checkpoint chosen by MIN val_loss) ===
  Total training time: 1710.57 s
  Best epoch: 2 / 4
  Best-epoch val inference_s: 14.93
  Micro F1:  0.4967
  Macro F1: 0.4043
  Per-label F1:
    toxic: 0.6531
    severe_toxic: 0.2742
    obscene: 0.6422
    threat: 0.0753
    insult: 0.5908
    identity_hate: 0.1903

--- Proposal summary (record for report / comparison) ---
  device: cpu
  training_time_s: 1710.57
  inference_time_s: 10.41
  num_parameters: 5122106

--- Run summary ---
  mode: full
  best_epoch: 2
  final_train_loss: 0.3466
  val_loss: 0.4035
  inference_time_s: 10.41
  best_epoch_val_inference_time_s: 14.93
  glove_coverage: 17881/50000 (35.76%)

=== Per-label

## Training-Size Benchmark

Controlled CNN benchmark across increasing training-set sizes. This section keeps the current final CNN setup fixed (`USE_POS_WEIGHT=True`, `USE_GLOVE=True`, same preprocessing/split/model) and trains a fresh model for exactly 2 epochs at each size. It does **not** use early stopping or best-epoch selection.

In [5]:
import json

if "best_epoch" not in globals():
    raise RuntimeError(
        "Run the full-dataset training cell first so benchmark epochs can be fixed to its best_epoch."
    )

FULL_DATA_BEST_EPOCH = int(best_epoch)
BENCHMARK_EPOCHS = FULL_DATA_BEST_EPOCH
BENCHMARK_EPOCH_POLICY = "fixed_to_full_dataset_best_epoch"
BENCHMARK_STEP = 10_000
BENCHMARK_PREPROCESSING_KWARGS = {
    "use_iterative_stratify": True,
    "rebalance_train": True,
    "clean_to_toxic_ratio": 3.0,
    "rare_labels": ("severe_toxic", "threat", "identity_hate"),
    "rare_oversample_factor": 2.0,
    "max_copies_per_row": 3,
    "rebalance_random_state": 42,
    "print_diagnostics": True,
}
BENCHMARK_THRESHOLDS = np.linspace(0.01, 0.99, 99)

if not USE_POS_WEIGHT:
    raise ValueError("Training-size benchmark expects USE_POS_WEIGHT=True for the current final CNN setup.")
if not USE_GLOVE:
    raise ValueError("Training-size benchmark expects USE_GLOVE=True for the current final CNN setup.")
if not Path(GLOVE_PATH).is_file():
    raise FileNotFoundError(f"GloVe file not found: {GLOVE_PATH}")


def benchmark_train_sizes(full_train_n: int, step: int = 10_000) -> list[int]:
    sizes = list(range(step, full_train_n, step))
    if not sizes or sizes[-1] != full_train_n:
        sizes.append(full_train_n)
    return sizes


def subset_pos_weight(y_subset: np.ndarray) -> torch.Tensor:
    y_np = np.asarray(y_subset, dtype=np.float64)
    pos = y_np.sum(axis=0)
    neg = y_np.shape[0] - pos
    weights = np.where(pos > 0, neg / pos, 1.0).astype(np.float32)
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


def init_benchmark_model(current_vocab_size: int, current_glove_matrix: np.ndarray) -> TextCNN:
    m = TextCNN(current_vocab_size, embed_dim, num_labels, padding_idx=0).to(DEVICE)
    with torch.no_grad():
        m.embedding.weight.copy_(torch.tensor(current_glove_matrix, dtype=torch.float32, device=DEVICE))
    m.embedding.weight.requires_grad = bool(GLOVE_TRAINABLE)
    return m


def validation_logits_and_loss(m: nn.Module, criterion: nn.Module) -> tuple[np.ndarray, float, float]:
    m.eval()
    logits_out = []
    infer_t0 = time.perf_counter()
    with torch.no_grad():
        for xb, _yb in benchmark_val_loader:
            xb = xb.to(DEVICE)
            logits_out.append(m(xb).cpu().numpy())
    inference_time_s = time.perf_counter() - infer_t0

    logits_np = np.concatenate(logits_out, axis=0)
    with torch.no_grad():
        logits_t = torch.tensor(logits_np, dtype=torch.float32, device=DEVICE)
        y_t = torch.tensor(y_val_arr, dtype=torch.float32, device=DEVICE)
        val_loss = float(criterion(logits_t, y_t).item())
    return logits_np, val_loss, inference_time_s


def tune_per_label_thresholds(y_true: np.ndarray, y_prob: np.ndarray) -> tuple[np.ndarray, dict[str, float]]:
    y_int = y_true.astype(np.int64)
    pred_tuned = np.zeros_like(y_int, dtype=np.int64)
    best_thresholds: dict[str, float] = {}
    for i, label in enumerate(LABEL_COLUMNS):
        yt = y_int[:, i]
        yp = y_prob[:, i]
        best_f, best_t = -1.0, 0.5
        for t in BENCHMARK_THRESHOLDS:
            pr = (yp >= t).astype(np.int64)
            f = sk_f1(yt, pr, zero_division=0)
            if f > best_f + 1e-12 or (
                abs(f - best_f) <= 1e-12 and abs(float(t) - 0.5) < abs(best_t - 0.5)
            ):
                best_f, best_t = float(f), float(t)
        best_thresholds[str(label)] = float(best_t)
        pred_tuned[:, i] = (yp >= best_t).astype(np.int64)
    return pred_tuned, best_thresholds


def rows_from_per_label(df: pd.DataFrame) -> list[dict[str, float | str]]:
    return [
        {
            "label": str(row["label"]),
            "precision": float(row["precision"]),
            "recall": float(row["recall"]),
            "f1": float(row["f1"]),
            "roc_auc": float(row["roc_auc"]),
        }
        for _, row in df.iterrows()
    ]


print("Preparing ALL ON full training split to determine benchmark sizes...")
benchmark_full_data = preprocess_for_cnn(
    max_len=MAX_LEN,
    validation_fraction=0.1,
    random_state=42,
    min_freq=MIN_FREQ,
    max_vocab=MAX_VOCAB,
    max_train_samples=None,
    max_val_samples=_val_n,
    **BENCHMARK_PREPROCESSING_KWARGS,
)
benchmark_sizes = benchmark_train_sizes(benchmark_full_data.X_train.shape[0], BENCHMARK_STEP)

benchmark_summary_rows = []
benchmark_details = []

print("Training-size benchmark sizes:", benchmark_sizes)
print("Benchmark epoch policy:", BENCHMARK_EPOCH_POLICY)
print("Full-dataset best epoch used for every train size:", FULL_DATA_BEST_EPOCH)
print("Preprocessing mode: ALL ON")
print("Setup:", f"USE_POS_WEIGHT={USE_POS_WEIGHT}", f"USE_GLOVE={USE_GLOVE}", f"BATCH_SIZE={BATCH_SIZE}", f"MAX_LEN={MAX_LEN}")

for train_size in benchmark_sizes:
    print()
    print("=" * 80)
    print(f"Preprocessing ALL ON benchmark train_size={train_size}")
    bench_data = preprocess_for_cnn(
        max_len=MAX_LEN,
        validation_fraction=0.1,
        random_state=42,
        min_freq=MIN_FREQ,
        max_vocab=MAX_VOCAB,
        max_train_samples=train_size,
        max_val_samples=_val_n,
        **BENCHMARK_PREPROCESSING_KWARGS,
    )
    x_subset = bench_data.X_train
    y_subset = bench_data.y_train
    y_val_arr = np.asarray(bench_data.y_val, dtype=np.float32)
    current_vocab_size = len(bench_data.vocab)
    current_glove_matrix, current_glove_matched, current_glove_total = build_glove_weight_matrix(
        bench_data.vocab, Path(GLOVE_PATH), embed_dim
    )
    benchmark_val_loader = DataLoader(
        TensorDataset(
            torch.tensor(bench_data.X_val, dtype=torch.long),
            torch.tensor(bench_data.y_val, dtype=torch.float32),
        ),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    bench_model = init_benchmark_model(current_vocab_size, current_glove_matrix)
    bench_loss_fn = nn.BCEWithLogitsLoss(pos_weight=subset_pos_weight(y_subset))
    bench_optimizer = torch.optim.Adam(bench_model.parameters(), lr=LR)
    train_loader_bench = DataLoader(
        TensorDataset(
            torch.tensor(x_subset, dtype=torch.long),
            torch.tensor(y_subset, dtype=torch.float32),
        ),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    train_t0 = time.perf_counter()
    last_train_loss = 0.0
    for epoch in range(BENCHMARK_EPOCHS):
        bench_model.train()
        epoch_loss = 0.0
        n_batches = 0
        for xb, yb in train_loader_bench:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            bench_optimizer.zero_grad()
            loss = bench_loss_fn(bench_model(xb), yb)
            loss.backward()
            bench_optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
        last_train_loss = epoch_loss / max(n_batches, 1)
    training_time_s = time.perf_counter() - train_t0

    logits_val, val_loss_bench, inference_time_s = validation_logits_and_loss(bench_model, bench_loss_fn)
    prob_val = torch.sigmoid(torch.from_numpy(logits_val)).numpy()

    pred_05 = (prob_val >= 0.5).astype(np.int64)
    per_label_05, summary_05 = multilabel_evaluation_report(y_val_arr, pred_05, prob_val, LABEL_COLUMNS)

    tune_t0 = time.perf_counter()
    pred_tuned, best_thresholds = tune_per_label_thresholds(y_val_arr, prob_val)
    threshold_tuning_time_s = time.perf_counter() - tune_t0
    per_label_tuned, summary_tuned = multilabel_evaluation_report(y_val_arr, pred_tuned, prob_val, LABEL_COLUMNS)

    f1_05 = {str(row["label"]): float(row["f1"]) for _, row in per_label_05.iterrows()}
    f1_tuned = {str(row["label"]): float(row["f1"]) for _, row in per_label_tuned.iterrows()}
    precision_05 = {str(row["label"]): float(row["precision"]) for _, row in per_label_05.iterrows()}
    recall_05 = {str(row["label"]): float(row["recall"]) for _, row in per_label_05.iterrows()}
    precision_tuned = {str(row["label"]): float(row["precision"]) for _, row in per_label_tuned.iterrows()}
    recall_tuned = {str(row["label"]): float(row["recall"]) for _, row in per_label_tuned.iterrows()}

    summary_row = {
        "train_size": int(train_size),
        "epochs": int(BENCHMARK_EPOCHS),
        "training_time_s": float(training_time_s),
        "inference_time_s": float(inference_time_s),
        "threshold_tuning_time_s": float(threshold_tuning_time_s),
        "train_loss": float(last_train_loss),
        "val_loss": float(val_loss_bench),
        "micro_f1": float(summary_05["f1_micro"]),
        "macro_f1": float(summary_05["f1_macro"]),
        "micro_macro_gap": float(summary_05["f1_micro"] - summary_05["f1_macro"]),
        "threat_f1": float(f1_05.get("threat", 0.0)),
        "identity_hate_f1": float(f1_05.get("identity_hate", 0.0)),
        "tuned_micro_f1": float(summary_tuned["f1_micro"]),
        "tuned_macro_f1": float(summary_tuned["f1_macro"]),
        "tuned_micro_macro_gap": float(summary_tuned["f1_micro"] - summary_tuned["f1_macro"]),
        "tuned_threat_f1": float(f1_tuned.get("threat", 0.0)),
        "tuned_identity_hate_f1": float(f1_tuned.get("identity_hate", 0.0)),
    }
    benchmark_summary_rows.append(summary_row)

    benchmark_details.append(
        {
            **summary_row,
            "threshold_0_5": {
                "f1_micro": float(summary_05["f1_micro"]),
                "f1_macro": float(summary_05["f1_macro"]),
                "per_label": rows_from_per_label(per_label_05),
                "per_label_f1": f1_05,
                "per_label_precision": precision_05,
                "per_label_recall": recall_05,
            },
            "threshold_tuned": {
                "f1_micro": float(summary_tuned["f1_micro"]),
                "f1_macro": float(summary_tuned["f1_macro"]),
                "best_threshold_per_label": best_thresholds,
                "per_label": rows_from_per_label(per_label_tuned),
                "per_label_f1": f1_tuned,
                "per_label_precision": precision_tuned,
                "per_label_recall": recall_tuned,
            },
        }
    )

    print(
        f"train_size={train_size} | train_s={training_time_s:.1f} | infer_s={inference_time_s:.1f} | "
        f"micro={summary_row['micro_f1']:.4f} macro={summary_row['macro_f1']:.4f} | "
        f"threat={summary_row['threat_f1']:.4f} identity={summary_row['identity_hate_f1']:.4f} | "
        f"tuned_micro={summary_row['tuned_micro_f1']:.4f} tuned_macro={summary_row['tuned_macro_f1']:.4f} | "
        f"tuned_threat={summary_row['tuned_threat_f1']:.4f} tuned_identity={summary_row['tuned_identity_hate_f1']:.4f}",
        flush=True,
    )

benchmark_df = pd.DataFrame(benchmark_summary_rows)
print()
print("Training-size benchmark summary:")
display(benchmark_df)

benchmark_out_dir = ROOT / "notebooks" / "cnn_baseline_outputs"
benchmark_out_dir.mkdir(parents=True, exist_ok=True)
benchmark_stem = f"cnn_glove_posweight_size_benchmark_fullds_bestepoch{FULL_DATA_BEST_EPOCH}_newprep_allon"
benchmark_csv = benchmark_out_dir / f"{benchmark_stem}_summary.csv"
benchmark_json = benchmark_out_dir / f"{benchmark_stem}_details.json"
benchmark_df.to_csv(benchmark_csv, index=False)
with open(benchmark_json, "w", encoding="utf-8") as f:
    json.dump(
        {
            "experiment": "training_size_benchmark",
            "setup": {
                "model": "TextCNN",
                "use_pos_weight": bool(USE_POS_WEIGHT),
                "use_glove": bool(USE_GLOVE),
                "glove": glove_stats,
                "epoch_policy": BENCHMARK_EPOCH_POLICY,
                "full_dataset_best_epoch": int(FULL_DATA_BEST_EPOCH),
                "epochs_per_size": int(BENCHMARK_EPOCHS),
                "batch_size": int(BATCH_SIZE),
                "max_len": int(MAX_LEN),
                "lr": float(LR),
                "validation_samples": int(benchmark_full_data.y_val.shape[0]),
                "train_sizes": [int(s) for s in benchmark_sizes],
                "preprocessing": {
                    "use_iterative_stratify": True,
                    "rebalance_train": True,
                    "clean_to_toxic_ratio": 3.0,
                    "rare_labels": ["severe_toxic", "threat", "identity_hate"],
                    "rare_oversample_factor": 2.0,
                    "max_copies_per_row": 3,
                    "rebalance_random_state": 42,
                    "print_diagnostics": True,
                },
            },
            "summary": benchmark_summary_rows,
            "details": benchmark_details,
        },
        f,
        indent=2,
    )
print("Saved:", benchmark_csv)
print("Saved:", benchmark_json)

Training-size benchmark sizes: [10000, 20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000, 100000, 110000, 120000, 130000, 140000, 143613]
Benchmark epoch policy: fixed_to_full_dataset_best_epoch
Full-dataset best epoch used for every train size: 2
Setup: USE_POS_WEIGHT=True USE_GLOVE=True BATCH_SIZE=64 MAX_LEN=256
train_size=10000 | train_s=55.5 | infer_s=15.4 | micro=0.3842 macro=0.3226 | threat=0.0418 identity=0.1374 | tuned_micro=0.5698 tuned_macro=0.4249 | tuned_threat=0.1261 tuned_identity=0.2238
train_size=20000 | train_s=113.5 | infer_s=13.9 | micro=0.4347 macro=0.3456 | threat=0.0610 identity=0.1510 | tuned_micro=0.5981 tuned_macro=0.4545 | tuned_threat=0.1351 tuned_identity=0.2390
train_size=30000 | train_s=171.9 | infer_s=16.2 | micro=0.4143 macro=0.3238 | threat=0.0760 identity=0.1413 | tuned_micro=0.6256 tuned_macro=0.4720 | tuned_threat=0.1391 tuned_identity=0.2654
train_size=40000 | train_s=219.0 | infer_s=12.5 | micro=0.4615 macro=0.3639 | threat=0.0741 identity=0.1

,train_size,epochs,training_time_s,inference_time_s,threshold_tuning_time_s,train_loss,val_loss,micro_f1,macro_f1,micro_macro_gap,threat_f1,identity_hate_f1,tuned_micro_f1,tuned_macro_f1,tuned_micro_macro_gap,tuned_threat_f1,tuned_identity_hate_f1
0,10000,2,55.510934,15.375785,2.153990,0.627741,0.628610,0.384205,0.322556,0.061650,0.041829,0.137405,0.569844,0.424931,0.144913,0.126126,0.223810
1,20000,2,113.466051,13.861687,1.915047,0.494621,0.562911,0.434700,0.345640,0.089060,0.060976,0.151023,0.598051,0.454500,0.143550,0.135135,0.238994
2,30000,2,171.937785,16.197326,1.972297,0.469154,0.575631,0.414322,0.323820,0.090501,0.076030,0.141343,0.625601,0.472030,0.153571,0.139130,0.265403
3,40000,2,218.960601,12.462356,1.873036,0.427168,0.536344,0.461504,0.363887,0.097617,0.074074,0.160615,0.640795,0.489834,0.150961,0.148148,0.286432
4,50000,2,285.249112,13.493484,1.982614,0.397539,0.579430,0.496659,0.381884,0.114774,0.088698,0.184697,0.645047,0.501165,0.143882,0.162437,0.296296
5,60000,2,345.013010,17.545494,3.153161,0.390789,0.496642,0.466598,0.366893,0.099705,0.105012,0.163209,0.663038,0.519276,0.143761,0.207407,0.325991
6,70000,2,425.078971,12.095213,1.906906,0.384210,0.494860,0.502658,0.399752,0.102906,0.093784,0.166786,0.670418,0.518953,0.151465,0.163743,0.328767
7,80000,2,452.676021,12.176758,1.889026,0.371381,0.429557,0.475795,0.385514,0.090281,0.072948,0.174879,0.677362,0.541998,0.135364,0.252252,0.352357
8,90000,2,532.404686,12.802084,1.923350,0.362346,0.439336,0.519016,0.421874,0.097142,0.131994,0.146002,0.681428,0.540048,0.141379,0.253521,0.327945
9,100000,2,598.350316,12.797688,1.891571,0.362041,0.422031,0.505779,0.403584,0.102195,0.095713,0.178730,0.686192,0.545487,0.140706,0.265193,0.336336


Saved: C:\Users\alici\Downloads\cmpe258-2026Fall-ToxicCommentDetection\notebooks\cnn_baseline_outputs\cnn_glove_posweight_size_benchmark_fullds_bestepoch2_newprep_alloff_summary.csv
Saved: C:\Users\alici\Downloads\cmpe258-2026Fall-ToxicCommentDetection\notebooks\cnn_baseline_outputs\cnn_glove_posweight_size_benchmark_fullds_bestepoch2_newprep_alloff_details.json


In [6]:
# Export training-size benchmark as one per-label reporting CSV (no retraining)
if "FULL_DATA_BEST_EPOCH" in globals():
    RUN_ID = f"cnn_glove_posweight_size_benchmark_fullds_bestepoch{int(FULL_DATA_BEST_EPOCH)}_newprep_allon"
else:
    RUN_ID = "cnn_glove_posweight_size_benchmark_fullds_bestepoch2_newprep_allon"
BENCHMARK_OUT_DIR = ROOT / "notebooks" / "cnn_baseline_outputs"
BENCHMARK_DETAILS_JSON = BENCHMARK_OUT_DIR / f"{RUN_ID}_details.json"
PER_LABEL_CSV = BENCHMARK_OUT_DIR / f"{RUN_ID}_per_label.csv"

# Prefer in-memory results if this notebook session just ran the benchmark; otherwise load saved details.
if "benchmark_details" in globals():
    _benchmark_details = benchmark_details
else:
    with open(BENCHMARK_DETAILS_JSON, "r", encoding="utf-8") as f:
        _benchmark_details = json.load(f)["details"]

_label_order = {label: i for i, label in enumerate(LABEL_COLUMNS)}
_rare_labels = {"threat", "identity_hate", "severe_toxic"}
rows = []
for item in _benchmark_details:
    baseline_by_label = {row["label"]: row for row in item["threshold_0_5"]["per_label"]}
    tuned_by_label = {row["label"]: row for row in item["threshold_tuned"]["per_label"]}

    for label in LABEL_COLUMNS:
        baseline = baseline_by_label[str(label)]
        tuned = tuned_by_label[str(label)]
        rows.append(
            {
                "run_id": RUN_ID,
                "embedding_variant": "glove",
                "loss_variant": "bce_pos_weight",
                "train_size": int(item["train_size"]),
                "epochs": int(item["epochs"]),
                "label": str(label),
                "val_loss": float(item["val_loss"]),
                "is_rare_label": str(label) in _rare_labels,
                "training_time_s": float(item["training_time_s"]),
                "inference_time_s": float(item["inference_time_s"]),
                "threshold_tuning_time_s": float(item["threshold_tuning_time_s"]),
                "micro_f1": float(item["micro_f1"]),
                "macro_f1": float(item["macro_f1"]),
                "tuned_micro_f1": float(item["tuned_micro_f1"]),
                "tuned_macro_f1": float(item["tuned_macro_f1"]),
                "micro_macro_gap": float(item["micro_macro_gap"]),
                "tuned_micro_macro_gap": float(item["tuned_micro_macro_gap"]),
                "precision_threshold_0_5": float(baseline["precision"]),
                "recall_threshold_0_5": float(baseline["recall"]),
                "f1_threshold_0_5": float(baseline["f1"]),
                "roc_auc_threshold_0_5": float(baseline["roc_auc"]),
                "precision_tuned": float(tuned["precision"]),
                "recall_tuned": float(tuned["recall"]),
                "f1_tuned": float(tuned["f1"]),
                "roc_auc_tuned": float(tuned["roc_auc"]),
                "_label_order": _label_order[str(label)],
            }
        )

_ordered_columns = [
    # meta
    "run_id",
    "embedding_variant",
    "loss_variant",
    "train_size",
    "epochs",
    "label",
    "is_rare_label",
    # efficiency
    "training_time_s",
    "inference_time_s",
    "threshold_tuning_time_s",
    # selection
    "val_loss",
    # overall - threshold=0.5
    "micro_f1",
    "macro_f1",
    "micro_macro_gap",
    # overall - tuned
    "tuned_micro_f1",
    "tuned_macro_f1",
    "tuned_micro_macro_gap",
    # per-label - threshold=0.5
    "precision_threshold_0_5",
    "recall_threshold_0_5",
    "f1_threshold_0_5",
    "roc_auc_threshold_0_5",
    # per-label - tuned
    "precision_tuned",
    "recall_tuned",
    "f1_tuned",
    "roc_auc_tuned",
]
benchmark_per_label_df = (
    pd.DataFrame(rows)
    .sort_values(["train_size", "_label_order"])
    .drop(columns=["_label_order"])
    .reset_index(drop=True)
)
benchmark_per_label_df = benchmark_per_label_df[_ordered_columns]

benchmark_per_label_df.to_csv(PER_LABEL_CSV, index=False)
print("Saved:", PER_LABEL_CSV)
display(benchmark_per_label_df)

Saved: C:\Users\alici\Downloads\cmpe258-2026Fall-ToxicCommentDetection\notebooks\cnn_baseline_outputs\cnn_glove_posweight_size_benchmark_fullds_bestepoch2_newprep_alloff_per_label.csv


,run_id,embedding_variant,loss_variant,train_size,epochs,label,is_rare_label,training_time_s,inference_time_s,threshold_tuning_time_s,...,tuned_macro_f1,tuned_micro_macro_gap,precision_threshold_0_5,recall_threshold_0_5,f1_threshold_0_5,roc_auc_threshold_0_5,precision_tuned,recall_tuned,f1_tuned,roc_auc_tuned
0,cnn_glove_posweight_size_benchmark_fullds_best...,glove,bce_pos_weight,10000,2,toxic,False,55.510934,15.375785,2.153990,...,0.424931,0.144913,0.464990,0.780013,0.582647,0.928957,0.605430,0.680078,0.640587,0.928957
1,cnn_glove_posweight_size_benchmark_fullds_best...,glove,bce_pos_weight,10000,2,severe_toxic,True,55.510934,15.375785,2.153990,...,0.424931,0.144913,0.098329,0.938650,0.178010,0.971691,0.269841,0.625767,0.377079,0.971691
2,cnn_glove_posweight_size_benchmark_fullds_best...,glove,bce_pos_weight,10000,2,obscene,False,55.510934,15.375785,2.153990,...,0.424931,0.144913,0.390058,0.803614,0.525197,0.947819,0.634328,0.614458,0.624235,0.947819
3,cnn_glove_posweight_size_benchmark_fullds_best...,glove,bce_pos_weight,10000,2,threat,True,55.510934,15.375785,2.153990,...,0.424931,0.144913,0.021468,0.811321,0.041829,0.939547,0.120690,0.132075,0.126126,0.939547
4,cnn_glove_posweight_size_benchmark_fullds_best...,glove,bce_pos_weight,10000,2,insult,False,55.510934,15.375785,2.153990,...,0.424931,0.144913,0.328434,0.827586,0.470247,0.944365,0.498992,0.632184,0.557746,0.944365
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,cnn_glove_posweight_size_benchmark_fullds_best...,glove,bce_pos_weight,143613,2,severe_toxic,True,886.140259,16.091444,1.953332,...,0.560899,0.134851,0.163712,0.963190,0.279857,0.985005,0.403042,0.650307,0.497653,0.985005
86,cnn_glove_posweight_size_benchmark_fullds_best...,glove,bce_pos_weight,143613,2,obscene,False,886.140259,16.091444,1.953332,...,0.560899,0.134851,0.485641,0.916867,0.634960,0.979762,0.855051,0.703614,0.771976,0.979762
87,cnn_glove_posweight_size_benchmark_fullds_best...,glove,bce_pos_weight,143613,2,threat,True,886.140259,16.091444,1.953332,...,0.560899,0.134851,0.069666,0.905660,0.129380,0.981615,0.192593,0.490566,0.276596,0.981615
88,cnn_glove_posweight_size_benchmark_fullds_best...,glove,bce_pos_weight,143613,2,insult,False,886.140259,16.091444,1.953332,...,0.560899,0.134851,0.412178,0.899106,0.565235,0.971069,0.624217,0.763729,0.686962,0.971069


## Proposal checklist (report)

- Compare **training time** and **parameter count** across the four model notebooks.
- **Per-label threshold tuning** on validation (default here: 0.5) for rare classes such as `threat`.
- **Model size on disk**: save `state_dict` or full checkpoint and report file size in MB.
- Optional: **macro** vs **micro** trade-offs given imbalance (already reported above).